##  Introduction


Welcome to this hands-on lab on **Audio Classification using Deep Learning**.

You'll use the [UrbanSound8K dataset](https://urbansounddataset.weebly.com/urbansound8k.html) to classify sounds from urban environments.

- 0 = air_conditioner
- 1 = car_horn
- 2 = children_playing
- 3 = dog_bark
- 4 = drilling
- 5 = engine_idling
- 6 = gun_shot
- 7 = jackhammer
- 8 = siren
- 9 = street_music

We will go through each step required to:
- Prepare and load audio data
- Build a PyTorch Dataset
- Transform waveforms into log-mel spectrograms
- Train a CNN to classify sounds into 10 classes


In [ ]:
import torch
import torchaudio
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn.functional as F
import torch.optim.lr_scheduler as lr_scheduler
import torch.optim as optim
from torchsummary import summary
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd
import pdb
import librosa
import librosa.display
import IPython.display as ipd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

## Define the arguments needed to instantiate the MyCustomDataset class and Open Metadata


We begin by loading the `UrbanSound8K.csv` metadata file. This file contains the filename, label, and which fold (subfolder) the file is stored in.

Use matplotlib to represent all the different labels. Comments


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Annotations file is the path to the UrbanSound8K.csv file

Audio Directory is the path to the folder containaing the 10 folds of audio files

We will be using the log-mel spectrogram transformation of torchaudio

Sample rate is 22050 and Number of samples is also 22050

Device is either GPU (mps for Apple and CUDA for Windows) or CPU

---



In [ ]:
ANNOTATIONS_FILE =
AUDIO_DIR =
SAMPLE_RATE =
NUM_SAMPLES =


if torch.cuda.is_available():
    device = "cuda"  # GPU
else:
    device = "cpu"   # Default to CP

print(f"We are using {device} device")

We are using cuda device


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv()

# Preview the first few rows of your .csv


In [ ]:
# Using matplotlib look at the number of sample per each categories

## Define Transforms (torchaudio)


We use torchaudio to define our transformation pipeline:
- Convert waveform into Mel Spectrogram
- Convert amplitude to decibels

These features work better with CNNs than raw audio waveforms.


In [ ]:
mel_spect = torchaudio.transforms.MelSpectrogram(

)

log_mel_spect = torchaudio.transforms.AmplitudeToDB()

transform =



## Audio Utility Class


We use a utility class: class AudioUtil

Create different function to

- resample
- pad
- rechannel (all in mono)

These steps ensure that all audio inputs are consistent in format and size.


In [ ]:
class AudioUtil:
    def open(self, audio_file):
        sig, sr = torchaudio.load(audio_file)
        return (sig, sr)

    def rechannel(self, aud, new_channel=1):


        return (resig, sr)

    def resample(self, aud, newsr):

        return ((resig, newsr))

    def pad_trunc(self, aud, max_ms):

        return (sig, sr)

    def spectro_gram(self, aud):

        return transform()

        return spec


##  Custom Dataset (MyCustomDataset)


We now define a custom Dataset class to:
- Load `.wav` files
- Apply transforms
- Return (tensor, label)

This is memory-efficient and PyTorch-compatible.


In [ ]:
class MyCustomDataset(Dataset):
    def __init__(self, annotation_file, audio_dir, transform, target_sample_rate, num_samples, device):
        self.annotations = pd.read_csv(annotation_file)
        self.audio_dir = audio_dir
        self.device = device
        self.transform = transform
        self.target_sample_rate = target_sample_rate
        self.num_samples = num_samples
        self.util = AudioUtil()

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):
        fold = f"fold{self.annotations.iloc[index, 5]}"
        file_name =
        audio_path = os.path.join()
        label =
        signal, sr = self.util.open()
        #processing
        signal, sr = self.util.resample()
        signal, sr = self.util.rechannel()
        signal, sr = self.util.pad_trunc()


        # Transform to spectrogram
        spec = self.transform(signal)

        return spec, label



### Split the dataset into training (80%), validation(10%), and test (10%) dataset
##### Use the random_split from torch.utils.data



In [ ]:
full_dataset = MyCustomDataset(
    annotation_file=,
    audio_dir=,
    transform=,
    target_sample_rate=,
    num_samples=,
    device=
)
total_size =
train_size =
val_size =
test_size =
train_dataset, val_dataset, test_dataset = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

##  Explore Dataset & Visualize


Now that we have a working dataset, let’s:
- Print the number of samples
- Retrieve the first and last samples
- Plot their log-Mel spectrograms


In [ ]:
# Print number of samples

# Retrieve first and last sample
first_spec, first_label =
last_spec, last_label =

print("First label:", first_label)
print("Last label:", last_label)


# Convert spectrograms from tensor to numpy
first_spec_np =
last_spec_np =

# Plot log-Mel spectrograms (first and last)

# First sample plot


# Last sample plot






## Dataloaders


### Create a train, validation and test dataloaders using dataloaders from torch.utils.data
##### Use a batch size of 64
##### Remember to turn on shuffle for training and turn it off for validation and testing


In [ ]:
BATCH_SIZE = 64

# Create DataLoaders
train_loader =

val_loader =

test_loader =

## Define CNN Model


The model structure will be unique for every student

The model should consist of convolutions, pooling, batch normalization, flatten, activation functions and linear layers

Recall that there are 10 classes in the dataset, hence the final layer will have 10 neurons


In [ ]:

class AudioClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(

        )

    def forward(self, x):
        return self.net(x)


Use the summary fuction in the torchsummary library to print the number of parameters and the estimated size of the model

In [ ]:
from torchsummary import summary
model = AudioClassifier().to(device)



## Train for 1 Epoch

Create a Training Function for one epoch only

Using a for loop, the train function will take as arguments: the model, train loader, loss function, optimizer, etc

Use the cross-entropy loss function and calculate accuracy


This helps check your model and dataset before full training.


In [ ]:

def train_one_epoch(model, loader, loss_fn, optimizer, device):





    acc = correct / total
    avg_loss = total_loss / len(loader)

    print(f"Train Accuracy: {acc:.4f} | Loss: {avg_loss:.4f}")
    return avg_loss, acc


### Run it for 1 epoch

In [ ]:
model = AudioClassifier().to(device)

# Define loss function and optimizer
loss_fn =
optimizer =

#Scheduler:adjusts the learning rate during training to help converge better or faster
scheduler =


# Train for one epoch
#train_one_epoch(model, train_loader, loss_fn, optimizer, device)

## Valid for 1 Epoch

Create a valid Function for one epoch only

##### Using a for loop, the validation function will take as arguments: the model, validation loader, loss function, scheduler
##### It will be wise to include best model parameters are arguments at this point
##### The goal is to monitor the performance of your proposed model to avoid overfitting/underfitting, know the best epoch to save the model weights, etc.

In [ ]:
def validate_one_epoch(model, val_loader, loss_fn, scheduler, best_val_loss, best_model_state, device):


    acc = correct / total
    avg_loss = total_loss / len(val_loader)

    print(f"Validation Accuracy: {acc:.4f} | Loss: {avg_loss:.4f}")

    # Step the scheduler
    if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):

    # Update best model
    if avg_loss < best_val_loss:


    return avg_loss, acc, best_val_loss, best_model_state


**IT IS TOO SLOW ON MY COMPUTER!! I will precompute all the melspectro and save them, so I do not do it for each epoch**

In [ ]:
def precompute_and_save(annotation_file, audio_dir, output_dir, transform, target_sample_rate, max_ms):
    os.makedirs(output_dir, exist_ok=True)
    df = pd.read_csv(annotation_file)

    util = AudioUtil()

    for idx in tqdm(range(len(df)), desc="Precomputing"):
        row = df.iloc[idx]
        fold = f"fold{row['fold']}"
        file_path = os.path.join(audio_dir, fold, row['slice_file_name'])
        label = row['classID']

        signal, sr = util.open(file_path)
        signal, sr = util.resample((signal, sr), target_sample_rate)
        signal, sr = util.rechannel((signal, sr), 1)
        signal, sr = util.pad_trunc((signal, sr), max_ms)

        spec = transform(signal)

        # Save spectrogram as tensor
        save_path = os.path.join(output_dir, f"{idx:05d}_{label}.pt")
        torch.save({'spec': spec, 'label': label}, save_path)

In [ ]:
precompute_and_save(
    annotation_file=ANNOTATIONS_FILE,
    audio_dir=AUDIO_DIR,
    output_dir="/content/drive/MyDrive/UrbanSound8K/precomputed",
    transform=transform,
    target_sample_rate=SAMPLE_RATE,
    max_ms=1000
)

Precomputing: 100%|██████████| 8732/8732 [1:20:45<00:00,  1.80it/s]


In [ ]:
class MycustomDataset_prepro(Dataset):
    def __init__(self, spectrogram_dir):
        self.files = sorted([
            os.path.join(spectrogram_dir, f)
            for f in os.listdir(spectrogram_dir) if f.endswith(".pt")
        ])


    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data = torch.load(self.files[idx], weights_only=False, map_location='cpu')

        spec = data['spec'].float()
        label = data['label']
        return spec, label

In [ ]:
full_dataset = MycustomDataset_prepro("/content/drive/MyDrive/UrbanSound8K/precomputed")

# Split into train/val/test
train_size =
val_size =
test_size = l
train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])

In [ ]:
BATCH_SIZE = 64

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
model =
loss_fn =
optimizer =
scheduler =


# Train for one epoch
#train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device)


## Full Training Loop

### Create a Function that will call the training and validation functions just created
##### This function will interate over all the epochs that you define
##### It will also activate your early stopping, so that the model does not continue training without improvements in the validation


In [ ]:

def full_train(
    model,
    train_loader,
    val_loader,
    loss_fn,
    optimizer,
    scheduler,
    device,
    num_epochs=30,
    patience=5
):



        # Train


        # Validate

        # Save metrics


        # Print metrics for epoch

        # Early Stopping logic


    print("\n Training complete.")
    return best_model_state, train_losses, val_losses, train_accuracies, val_accuracies


# Now commence training.
### At the point, you will manually revisit the model function to tune the hyperparameters with a goal of improving the results
##### Some of the hyperparameters to be tuned include
###### Learning rate
###### Number of convolution layers
###### Number of channels per layer
###### Number of neurons in the fully connected layer
###### Choice of activation function
###### Type of scheduler
###### Regularizers such as dropouts, L1, or L2 regularizers, etc.


### When you arrive at the most optimal performance, save the model weights at the points


In [ ]:
best_model_state, train_losses, val_losses, train_accs, val_accs=full_train(
    model,
    train_loader,
    val_loader,
    loss_fn,
    optimizer,
    scheduler,
    device,
    num_epochs=30,
    patience=20 #if validation loss hasn’t improved for patience epoch, stop
)

torch.save(best_model_state, 'best_model.pth')


### Make a plot of the training and validation losses as well as the accuracies over all the epochs considered

In [ ]:
def plot_training_curves(train_losses, val_losses, train_accs, val_accs):


In [ ]:
model = AudioClassifier().to(device)

# Load the best model weights

# set to evaluation model


In [ ]:
plot_training_curves(train_losses, val_losses, train_accs, val_accs)

### Evaluate ther performance using Confusion Matrix and Classification Report
##### The sklearn library provides functions that can be used to visualize these results
##### Give a detailed explanation of your results. Explain the concepts of accuracy, precision, recall, F1-Score, etc

In [ ]:
def get_all_preds_and_labels(model, loader, best_model_state, device):


In [ ]:
class_names = [
    "air_conditioner", "car_horn", "children_playing", "dog_bark", "drilling",
    "engine_idling", "gun_shot", "jackhammer", "siren", "street_music"
]

import seaborn as sns


In [ ]:
# Classification Report
print("Classification Report:")
print(classification_report(y_true, y_pred, digits=4))